# Day 069 — Solution: Image Extractor

In [ ]:
_EXTRACTOR_SRC = '"""image_extractor.py — Day 069: Multimodal Extraction.\n\nExtracts structured data from images by combining a vision LLM (Ollama llava)\nwith a Pydantic v2 schema. Returns a validated Python object.\n\nSetup:\n    ollama pull llava    # vision model\n\nUsage:\n    from image_extractor import ImageExtractor\n    from pydantic import BaseModel\n    from PIL import Image\n\n    class ProductInfo(BaseModel):\n        name: str\n        price: float\n        category: str = ""\n\n    extractor = ImageExtractor(schema_cls=ProductInfo)\n    img = Image.open("product.jpg")\n    result = extractor.extract(img)\n    print(result.name, result.price)\n\nTesting without Ollama:\n    mock = lambda b64, prompt: \'{"name": "Widget", "price": 9.99}\'\n    extractor = ImageExtractor(schema_cls=ProductInfo, describe_fn=mock)\n"""\nimport io\nimport re\nimport json\nimport base64\nfrom typing import Type, TypeVar\nfrom pydantic import BaseModel\n\nT = TypeVar("T", bound=BaseModel)\n\n\ndef image_to_base64(img, format: str = "PNG") -> str:\n    """Encode a PIL Image as a base64 string."""\n    buf = io.BytesIO()\n    out = img\n    if format.upper() in ("JPEG", "JPG") and img.mode in ("RGBA", "P"):\n        out = img.convert("RGB")\n    out.save(buf, format=format)\n    return base64.b64encode(buf.getvalue()).decode()\n\n\ndef build_extraction_prompt(schema_cls: Type[BaseModel]) -> str:\n    """Build a vision LLM prompt for structured JSON extraction.\n\n    Embeds the Pydantic model\'s JSON schema so the model knows exactly\n    which fields to return and their types.\n    """\n    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)\n    return (\n        "Extract structured data from this image and return ONLY valid JSON "\n        "matching this schema exactly. Do not include any explanation, "\n        "markdown, or code blocks — just the raw JSON object.\\n\\n"\n        f"Schema:\\n{schema_json}\\n\\n"\n        "Return ONLY the JSON object, nothing else."\n    )\n\n\ndef strip_json_from_response(response: str) -> str:\n    """Extract a JSON object from an LLM response string.\n\n    Handles:\n    - Plain JSON: {"key": "value"}\n    - Markdown code block: ```json\\n{...}\\n```\n    - JSON preceded by explanation text\n\n    Raises:\n        ValueError if no JSON object found in the response\n    """\n    # Try markdown code block first (```json ... ``` or ``` ... ```)\n    block = re.search(r"```(?:json)?\\s*([\\s\\S]*?)```", response)\n    if block:\n        return block.group(1).strip()\n\n    # Fall back to first {...} span in the string\n    obj = re.search(r"\\{[\\s\\S]*\\}", response)\n    if obj:\n        return obj.group(0).strip()\n\n    raise ValueError(\n        f"No JSON object found in LLM response: {response[:200]!r}"\n    )\n\n\ndef extract_from_image(img_b64: str, schema_cls: Type[T],\n                       describe_fn=None) -> dict:\n    """Send an image to a vision LLM and parse its response as JSON.\n\n    Args:\n        img_b64:    base64-encoded image string\n        schema_cls: Pydantic model class defining the target schema\n        describe_fn: callable(img_b64, prompt) -> str for testing\n    Returns:\n        dict of extracted fields (not yet validated against schema)\n    Raises:\n        ValueError if the response contains no parseable JSON\n        json.JSONDecodeError if the extracted JSON is malformed\n    """\n    prompt = build_extraction_prompt(schema_cls)\n    if describe_fn is not None:\n        response = describe_fn(img_b64, prompt)\n    else:\n        import ollama\n        resp = ollama.chat(\n            model="llava",\n            messages=[{\n                "role":    "user",\n                "content": prompt,\n                "images":  [img_b64],\n            }],\n        )\n        response = resp["message"]["content"]\n    raw = strip_json_from_response(response)\n    return json.loads(raw)\n\n\ndef safe_extract(img_b64: str, schema_cls: Type[T],\n                 describe_fn=None, retries: int = 2) -> dict | None:\n    """Extract JSON from an image, retrying on parse failures.\n\n    Args:\n        img_b64:    base64-encoded image string\n        schema_cls: Pydantic model class\n        describe_fn: callable for testing\n        retries:    number of extra attempts on failure (total = retries + 1)\n    Returns:\n        Extracted dict, or None if all attempts fail\n    """\n    for attempt in range(retries + 1):\n        try:\n            return extract_from_image(img_b64, schema_cls,\n                                      describe_fn=describe_fn)\n        except (ValueError, json.JSONDecodeError):\n            if attempt == retries:\n                return None\n    return None\n\n\ndef validate_extraction(data: dict, schema_cls: Type[T]) -> tuple:\n    """Validate an extracted dict against a Pydantic schema.\n\n    Args:\n        data:       dict from extract_from_image / safe_extract\n        schema_cls: Pydantic model class\n    Returns:\n        (True, validated_model) on success\n        (False, error_message_str) on validation failure\n    """\n    try:\n        model = schema_cls.model_validate(data)\n        return (True, model)\n    except Exception as exc:\n        return (False, str(exc))\n\n\nclass ImageExtractor:\n    """Extract structured Pydantic objects from images using a vision LLM.\n\n    Pass describe_fn for testing without Ollama::\n\n        mock = lambda b64, prompt: \'{"name": "Widget", "price": 9.99}\'\n        extractor = ImageExtractor(schema_cls=ProductInfo, describe_fn=mock)\n    """\n\n    def __init__(self, schema_cls: Type[T],\n                 model: str = "llava",\n                 describe_fn=None) -> None:\n        self.schema_cls  = schema_cls\n        self.model       = model\n        self._describe_fn = describe_fn\n\n    def extract(self, img, retries: int = 2) -> T:\n        """Extract and validate structured data from a PIL Image.\n\n        Encodes the image, calls the vision LLM, strips JSON from the\n        response, parses, and validates against schema_cls.\n\n        Args:\n            img:     PIL Image\n            retries: retry attempts on parse failure\n        Returns:\n            Validated Pydantic model instance\n        Raises:\n            ValueError if extraction fails after all retries\n            pydantic.ValidationError if the extracted data does not\n            match the schema after successful JSON parsing\n        """\n        img_b64 = image_to_base64(img)\n        data = safe_extract(img_b64, self.schema_cls,\n                            describe_fn=self._describe_fn,\n                            retries=retries)\n        if data is None:\n            raise ValueError(\n                f"Failed to extract valid JSON after {retries + 1} attempts"\n            )\n        return self.schema_cls.model_validate(data)\n'
from pathlib import Path
Path('image_extractor.py').write_text(_EXTRACTOR_SRC, encoding='utf-8')
print('image_extractor.py written.')

In [ ]:
import json, re, base64, io
from PIL import Image
from pydantic import BaseModel
from image_extractor import (
    ImageExtractor, image_to_base64, build_extraction_prompt,
    strip_json_from_response, extract_from_image, safe_extract,
    validate_extraction,
)

class ProductInfo(BaseModel):
    name:     str
    price:    float
    category: str = ''

_mock = lambda b64, prompt: '{"name": "Headphones", "price": 49.99, "category": "Electronics"}'
extractor = ImageExtractor(schema_cls=ProductInfo, describe_fn=_mock)
img = Image.new('RGB', (300, 150), 'white')

# 1. build_extraction_prompt embeds schema fields
prompt = build_extraction_prompt(ProductInfo)
assert 'name' in prompt and 'price' in prompt
print("\u2705 build_extraction_prompt contains field names")

# 2. strip_json_from_response handles markdown
raw = strip_json_from_response('```json\n{"a": 1}\n```')
assert json.loads(raw) == {"a": 1}
print("\u2705 strip_json_from_response handles markdown code block")

# 3. extract_from_image returns dict
d = extract_from_image(image_to_base64(img), ProductInfo, describe_fn=_mock)
assert isinstance(d, dict) and d['name'] == 'Headphones'
print("\u2705 extract_from_image returns dict")

# 4. safe_extract returns dict on success
d2 = safe_extract(image_to_base64(img), ProductInfo, describe_fn=_mock)
assert d2 is not None and d2['price'] == 49.99
print("\u2705 safe_extract returns dict on success")

# 5. safe_extract returns None on failure
d3 = safe_extract(image_to_base64(img), ProductInfo,
                   describe_fn=lambda b, p: 'no json', retries=1)
assert d3 is None
print("\u2705 safe_extract returns None after exhausted retries")

# 6. validate_extraction success
ok, model = validate_extraction({'name': 'Widget', 'price': 9.99}, ProductInfo)
assert ok and model.name == 'Widget'
print("\u2705 validate_extraction success: (True, model)")

# 7. validate_extraction failure
ok2, err = validate_extraction({'price': 9.99}, ProductInfo)
assert not ok2 and isinstance(err, str)
print("\u2705 validate_extraction failure: (False, error_str)")

# 8. ImageExtractor.extract returns typed model
result = extractor.extract(img)
assert isinstance(result, ProductInfo)
assert result.name == 'Headphones' and result.price == 49.99
print(f"\u2705 ImageExtractor.extract returns {type(result).__name__}: {result.name} ${result.price}")

print("\nMultimodal Extraction complete!")
